# GAT-GRU v03 — edge-feature factorial ablation

Chạy đủ 2³ tổ hợp của call_rate, call_ratio và edge_latency_p95_ms. Chỉ dùng train/validation.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / 'modeling-common').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'modeling-common'))
import forecast_ablation as fa

ctx = fa.load_context(ROOT)
selection = fa.load_selection_contract(ROOT / 'configs' / 'gatgru' / 'feature_contract_v01.json')
cfg = fa.load_baseline_config(ctx)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = ROOT / 'artifacts' / 'gatgru' / 'v03_edge_ablation'
screening = fa.run_fixed_variants(
    ctx, 'v03_edge_factorial', OUT,
    fa.edge_factorial_variants(cfg, selection['node_features']),
    seeds=(0, 1, 2), device=DEVICE, score_weights=fa.BASELINE_SCORE_WEIGHTS,
    parent_version='v02_graph_ablation',
)
display(screening)

,variant,model_kind,node_features,edge_features,edge_control,topology_control,n_seeds,primary_score_mean,primary_score_std,cpu_weighted_diagnostic_mean,...,cpu_cores_mae_macro_mean,cpu_cores_mae_macro_std,train_seconds_mean,train_seconds_std,early_stop_val_mae_mean,early_stop_val_mae_std,n_params_mean,n_params_std,epochs_trained_mean,epochs_trained_std
7,edge_call_rate__call_ratio__edge_latency_p95_ms,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,3,0.113974,0.001562,0.106518,...,0.013598,0.000296,625.639277,85.981220,0.113949,0.001561,121560.0,0.0,27.333333,2.516611
4,edge_call_rate,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...",[call_rate],real,real,3,0.114297,0.000959,0.106836,...,0.013608,0.000202,1000.582419,266.160239,0.114274,0.000960,121544.0,0.0,30.666667,4.725816
2,edge_call_ratio,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...",[call_ratio],real,real,3,0.114346,0.000895,0.106854,...,0.013600,0.000186,877.646470,297.386264,0.114324,0.000896,121544.0,0.0,30.666667,4.725816
0,edge_none,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...",[],real,real,3,0.114523,0.000858,0.107061,...,0.013673,0.000240,567.416366,102.666912,0.114500,0.000858,121528.0,0.0,27.666667,3.055050
6,edge_call_rate__call_ratio,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio]",real,real,3,0.114708,0.002667,0.107123,...,0.013682,0.000339,547.981148,98.498432,0.114684,0.002665,121552.0,0.0,23.666667,4.163332
3,edge_call_ratio__edge_latency_p95_ms,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_ratio, edge_latency_p95_ms]",real,real,3,0.114839,0.002162,0.107227,...,0.013682,0.000278,745.538397,125.227571,0.114815,0.002160,121552.0,0.0,23.666667,4.163332
5,edge_call_rate__edge_latency_p95_ms,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, edge_latency_p95_ms]",real,real,3,0.114960,0.002063,0.107354,...,0.013708,0.000240,531.548433,88.026452,0.114935,0.002061,121552.0,0.0,23.666667,4.163332
1,edge_edge_latency_p95_ms,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...",[edge_latency_p95_ms],real,real,3,0.114999,0.001463,0.107479,...,0.013733,0.000286,581.119629,41.453212,0.114974,0.001462,121544.0,0.0,23.666667,4.163332


## Five-seed confirmation

Xác nhận top ba tổ hợp và luôn giữ no-edge/all-edge anchors. Không tự động approve edge contract.

In [2]:
top = screening.head(3)
anchors = screening[screening.variant.isin(['edge_none', 'edge_call_rate__call_ratio__edge_latency_p95_ms'])]
finalists = pd.concat([top, anchors], ignore_index=True).drop_duplicates('variant')
specs = [dict(name='confirm_' + row.variant, kind='gatgru', config=cfg,
              node_features=selection['node_features'], edge_features=list(row.edge_features))
         for row in finalists.itertuples()]
confirmation = fa.run_fixed_variants(
    ctx, 'v03_edge_finalist_confirmation', OUT / 'finalists', specs,
    seeds=(0, 1, 2, 3, 4), device=DEVICE, score_weights=fa.BASELINE_SCORE_WEIGHTS,
    parent_version='v02_graph_ablation',
)
display(confirmation)

,variant,model_kind,node_features,edge_features,edge_control,topology_control,n_seeds,primary_score_mean,primary_score_std,cpu_weighted_diagnostic_mean,...,cpu_cores_mae_macro_mean,cpu_cores_mae_macro_std,train_seconds_mean,train_seconds_std,early_stop_val_mae_mean,early_stop_val_mae_std,n_params_mean,n_params_std,epochs_trained_mean,epochs_trained_std
0,confirm_edge_call_rate__call_ratio__edge_laten...,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,5,0.113994,0.001107,0.106535,...,0.013603,0.000213,836.890172,115.244519,0.113970,0.001106,121560.0,0.0,27.4,1.816590
3,confirm_edge_none,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...",[],real,real,5,0.114503,0.000825,0.107162,...,0.013711,0.000273,655.167102,47.941418,0.114481,0.000825,121528.0,0.0,27.6,2.190890
1,confirm_edge_call_rate,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...",[call_rate],real,real,5,0.114605,0.000810,0.107103,...,0.013642,0.000150,817.828035,343.325525,0.114583,0.000810,121544.0,0.0,29.8,3.563706
2,confirm_edge_call_ratio,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...",[call_ratio],real,real,5,0.114714,0.000852,0.107293,...,0.013692,0.000192,721.258690,171.909717,0.114692,0.000853,121544.0,0.0,28.6,5.029911
